# WaterSoftHack 2026 — Day 2: Edge Computing (Hands-On)

**Welcome back!** Yesterday you rented a cloud supercomputer and did all the thinking there.
Today we go to the other end of the wire: **inside the gauge standing in the river**.

You will program the *brain* of a solar-powered smart stream gauge and storm-test it against a
naive design. The gauge is simulated — same Python, same JupyterHub as yesterday, no soldering —
but every number (battery, bytes, timing) uses real field hardware values.

## What you will do in the next hour

1. Build a two-week river record with one nasty flash flood in it.
2. Meet the **virtual gauge**: a battery, a sensor, a radio, and *your* code in the middle.
3. Meter the **naive streamer** - the design that transmits every reading.
4. Program the **smart gauge** - detection rules, adaptive sampling, hourly summaries, a local buffer.
5. **Cut the network in the middle of the flood** and see which design still warns the town.
6. Read the final scoreboard: messages, bytes, battery life, and warning time.

> Run each cell with **Shift+Enter**. The lecture's slogan applies everywhere here:
> **it is cheaper to think than to talk.**

---
## Part 0 — The river we will test against

A real river is boring most of the time - which is exactly the point of edge computing, but it makes
a bad *test*. So we replay a **design storm**: a two-week, minute-by-minute stage record with one
flash flood in it (quiet baseflow, then a steep rise on day 10, crest above flood stage, slow recession).

If the room's network allows, the optional cell afterwards also pulls **live** USGS stage data so you
can see what today's actual Iowa River looks like - but the storm test below is deterministic, so
everyone gets the same scoreboard.

**Look at the shape.** Thirteen and a half days of "still about 2 feet, still fine", then a rise of
roughly **2 feet per hour** - a genuine flash flood signature. Keep two questions in mind:
how many of those ~20,000 one-minute readings are worth a radio transmission, and what happens if the
network dies during the interesting part?

### Optional: what does the real river say right now?

This cell asks the USGS API for live stage data at Iowa City (it falls back gracefully if the
network is busy). Not used in the storm test - just to connect the simulator to reality.

---
## Part 1 — The virtual gauge

Here is the hardware we are simulating, with realistic numbers for an ESP32-class station:

| Component | Draws | For | Cost per use |
|---|---|---|---|
| Deep sleep | 0.01 mA | always | ~0.25 mAh/**day** |
| Stage sensor read | 5 mA | 0.05 s | ~0.00007 mAh |
| Brain thinking | 40 mA | 0.01 s | ~0.0001 mAh |
| **Radio transmission** | **220 mA** | **2 s** | **~0.12 mAh** |

One battery: **10,000 mAh**. One radio message: **60 bytes**. Notice the radio costs roughly
**a thousand sensor readings** per message. That asymmetry is the whole game.

The `Gauge` class below is the simulation engine. It gives your code three abilities -
`read_sensor()`, `transmit(...)`, `local_alarm(...)` - and it keeps the honest ledger:
every microamp-hour, every byte, every message that failed because the network was down.
A gauge brain is a subclass that implements one method: `step(i, t)`, called once per simulated minute.

---
## Part 2 — Design A: the naive streamer

The obvious design, and exactly what we built yesterday: read the sensor every minute, transmit
every reading, let the cloud do all the thinking. Three lines of brain.

**Meter check.** 1,440 messages a day, ~84 KB a day, and the battery math: at roughly
175 mAh/day, a 10,000 mAh battery is gone in about **two months** - then someone drives to the
canyon with a fresh battery, forever. Also notice *what* it sent: 20,000 messages that almost all
say "still 2 feet, still fine."

On a healthy network this design does detect the flood promptly (the cloud sees every reading,
one minute late). Hold that thought - we will revisit it when the network stops being healthy.

---
## Part 3 — Design B: program the smart gauge

Now the brain from the lecture, the one you own. The behavior specification:

1. **Adaptive sampling** - read every **15 min** in quiet water, every **1 min** in event mode.
2. **Detection rules**, run on every sample, *on the device*:
   - `RAPID RISE` - the stage climbed more than **1.5 ft/hour** -> enter event mode, sound the alarm.
   - `FLOOD STAGE` - the stage crossed **10 ft** -> alarm again (the big one).
3. **The data diet** - one **hourly summary** message (min/mean/max); in event mode also a
   **situation report every 30 min**. Nothing else goes on the radio.
4. **Store-and-forward** - every reading is logged locally; if a transmission fails it is
   buffered and backfilled when the link returns.
5. Leave event mode when the water is back below 8 ft and falling.

The thresholds are the knobs at the top - after the storm test, come back and tune them.

**Read the two rows slowly.** Same river, same battery, same radio - the only difference is the
brain. The smart gauge sends **~25 messages a day instead of 1,440**, about **1.5 KB instead of
84 KB**, and the battery stops being the reason you drive to the canyon: it now outlives the
hardware around it. The alarms fired too - check:

---
## Part 4 — The storm test: cut the network mid-flood

Now the drill the lecture promised. Storms attack the network first, so: **the cellular link goes
down one hour into the rise and stays down for 12 hours** - right through the flood-stage crossing
and the crest. This is not pessimism; it is the correlated failure the reliability slide warned about.

Same two brains. One question: **does the town get warned?**

**This is the whole day in one printout.**

- The **naive streamer** kept firing its radio into a dead network. Its flood readings were not
  buffered, so they are *gone*: the cloud record has a 12-hour hole exactly where the flood was, and
  by the time the link returned the river had already receded below flood stage. No local alarm
  exists in that design. **The town found out from the water.**
- The **smart gauge** did not care that the network was down: the rules run on the device, so the
  siren on the bridge fired hours before flood stage, on schedule. Every reading it wanted to send
  was buffered, and after the link returned it backfilled - the cloud record *healed itself*, gap-free.

Efficiency was nice. **Autonomy is the point.**

---
## Part 5 — The final scoreboard

Everything on one table - and these are the same numbers you saw on the lecture's payoff slide,
because that slide *is* this simulator.

---
## Wrap-up

In one hour, on a simulated-but-honest smart gauge, you:

- replayed a two-week record with a flash flood through two gauge designs,
- metered the naive streamer: **1,440 msgs/day, ~84 KB/day, ~2 months of battery**,
- programmed on-device intelligence: two detection rules, adaptive sampling, hourly summaries,
- cut the network mid-flood and watched your design **still warn the town** and **heal the record**,
- and confirmed the lecture's arithmetic: **~25 msgs/day, ~1.5 KB/day, years of battery.**

### Looking ahead to Thursday (Hybrid)

Your smart gauge is a brilliant hermit: it knows its reach perfectly and nothing else. It cannot see
the rain upstream, the basin filling, or six hours into the future - that takes the cloud model you
built on Tuesday. Thursday we wire the two together: **edge triage, cloud forecasting, one pipeline.**
Bring both notebooks.

---
## Optional challenges (if you finish early)

1. **Tune the trigger.** Set `RISE_ALERT_FT_PER_HR = 0.5` and re-run Parts 3-4. How much earlier is
   the first alarm? What is the cost (messages, battery, false-alarm risk on the daily wiggle)?
2. **Starve the gauge.** Drop the battery to 2,000 mAh in `HW`. Which design survives the two weeks?
3. **Longer outage.** Make the outage 48 hours. Does the smart gauge's backfill still heal the record?
   (Check `len(smart_o.buffer)` - what would you change so the buffer cannot overflow in a real device?)
4. **A sneakier flood.** In `design_storm`, make the rise take 24 hours instead of 8 (change `peak`).
   Does the rate rule still fire before flood stage? What rule would you add?
5. **Replay reality.** If the live fetch worked, resample `live` to 1 minute
   (`live.resample('1min').interpolate()`) and run your `SmartGauge` on it. How many messages per day
   does a *real* quiet week cost?